In [2]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv()

SERVICE_KEY = os.getenv("SERVICE_KEY")
URL = "https://apis.data.go.kr/B551015/API186_1/SeoulRace_1"

def safe_get_items(data):
    try:
        body = data.get("response", {}).get("body", {})
        items = body.get("items", {})

        if not items:
            return []

        item = items.get("item", None)


        if item is None or item == "":
            return []

        if isinstance(item, dict):
            return [item]

        elif isinstance(item, list):
            return item

        else:
            return []

    except:
        return []

# ----------------------
# API 호출 (page 포함)
# ----------------------
def get_data(start_date, end_date, page):
    params = {
        "ServiceKey": SERVICE_KEY,
        "pageNo": str(page),
        "numOfRows": "1000",   # ⭐ 안정값
        "rc_date_fr": start_date,
        "rc_date_to": end_date,
        "_type": "json"
    }

    response = requests.get(URL, params=params)

    if response.status_code != 200:
        print("요청 실패:", response.status_code)
        return [], 0

    try:
        data = response.json()
        total = data.get("response", {}).get("body", {}).get("totalCount", 0)
    except:
        print("JSON 실패")
        return [], 0

    items = safe_get_items(data)
    return items, total

# ----------------------
# 🔄 날짜 + 페이지 병행
# ----------------------
start = datetime(2021, 1, 1)
end = datetime(2022, 12, 31)

all_data = []
current = start

while current <= end:
    next_date = current + timedelta(days=30)   # ⭐ 핵심 수정

    print(f"{current.date()} ~ {next_date.date()}")

    # 첫 페이지
    items, total = get_data(
        current.strftime("%Y%m%d"),
        next_date.strftime("%Y%m%d"),
        1
    )

    all_data.extend(items)

    page_size = 500
    total_pages = (total // page_size) + 1

    for page in range(2, total_pages + 1):
        items, _ = get_data(
            current.strftime("%Y%m%d"),
            next_date.strftime("%Y%m%d"),
            page
        )
        all_data.extend(items)

    current = next_date

# ----------------------
# 결과
# ----------------------
df = pd.DataFrame(all_data)

print("총 데이터 개수:", len(df))
print(df.head())

df.to_csv("raw_race_2021_to_2022.csv", index=False)

2021-01-01 ~ 2021-01-31
요청 실패: 401
2021-01-31 ~ 2021-03-02
요청 실패: 401
2021-03-02 ~ 2021-04-01
요청 실패: 401
2021-04-01 ~ 2021-05-01
요청 실패: 401
2021-05-01 ~ 2021-05-31
요청 실패: 401
2021-05-31 ~ 2021-06-30
요청 실패: 401
2021-06-30 ~ 2021-07-30
요청 실패: 401
2021-07-30 ~ 2021-08-29
요청 실패: 401
2021-08-29 ~ 2021-09-28
요청 실패: 401
2021-09-28 ~ 2021-10-28
요청 실패: 401
2021-10-28 ~ 2021-11-27
요청 실패: 401
2021-11-27 ~ 2021-12-27
요청 실패: 401
2021-12-27 ~ 2022-01-26
요청 실패: 401
2022-01-26 ~ 2022-02-25
요청 실패: 401
2022-02-25 ~ 2022-03-27
요청 실패: 401
2022-03-27 ~ 2022-04-26
요청 실패: 401
2022-04-26 ~ 2022-05-26
요청 실패: 401
2022-05-26 ~ 2022-06-25


KeyboardInterrupt: 